# CS 696 · Turn a process description into a what-if tool

Translate a short procedure into a traceable simulator, issue a controlled intervention, and explain which parts of the answer come from source clauses and which come from execution.

Module 02 linked each proposed transformation to a source clause and recorded unresolved decisions. Take that same evidence ledger into a new setting. This time the target field is a simulator input or rule. A missing rule limits the questions your tool can answer.

A fictional museum sends image trays through scanning and labeling. P1 says both stations open at minute zero and trays are processed in arrival order, breaking ties by the given list order. P2 gives one scanner that starts the next waiting tray as soon as it is free and takes four minutes per tray. P3 gives one labeler and seven minutes of labeling per tray, with work starting as soon as the scan and labeler are both ready. All durations are fixed. Times below are minutes since opening.

## Translate a clause into a start time

The archive can answer how long labeling normally takes by looking up P3. It cannot simply look up when a tray would finish if labeling took three minutes. For that, you need a rule about waiting. Start with two trays arriving at minutes 0 and 2. Their scans finish at 4 and 8. At minute 8 the second scan is ready, but the labeler is still working on the first tray until minute 11. Trace that handoff before writing a loop.

Lᵢ = max(Sᵢ, Fᵢ₋₁)     Fᵢ = Lᵢ + b

Sᵢ is the finish of tray i’s scan, Lᵢ the start of its labeling, and Fᵢ the finish of its labeling. F₀ = 0 is the station’s opening time. b = 7 is the labeling duration in P3. The maximum enforces both readiness conditions.

In [1]:
scan_ends = [4, 8]  # Hand-computed from arrivals 0, 2 and P1–P2.
label_free = 0
intervals = []
for scan_end in scan_ends:
    start = max(scan_end, label_free)
    label_free = start + 7
    intervals.append((start, label_free))
assert intervals == [(4, 11), (11, 18)]
print("Labeling intervals:", intervals)

Labeling intervals: [(4, 11), (11, 18)]


**Discuss** · If an implementation labels the second tray from minute 8 to minute 15, which clause would you show its author?

P3 assigns a single labeler. Starting tray 2 at minute 8 overlaps tray 1’s labeling interval from 4 to 11. A citation to the seven-minute duration alone would miss the error.

## Keep the intermediate events in the tool output

Add the scanner and keep one row per tray. The six columns are tray number, arrival, scan start, scan end, label start and label end. This function is supplied by the instructor. A language model could propose a function, but the research question here is whether a supplied tool answers the process question correctly. Check a hand-calculated case against the implementation before trusting its answer to a new question. A trace makes a wrong translation easier to locate than a single final number.

Aᵢ = max(aᵢ, Bᵢ₋₁)     Bᵢ = Aᵢ + s     Lᵢ = max(Bᵢ, Fᵢ₋₁)

aᵢ is the arrival of tray i, Aᵢ and Bᵢ are the start and end of its scan, and s is the per-tray scan duration. B₀ = F₀ = 0. Each station can work on a different tray at the same time. The final result is the last tray’s Fᵢ.

In [2]:
def simulate(arrivals, scan=4, label=7):
    if not arrivals or any(t < 0 for t in arrivals):
        raise ValueError("Provide nonnegative arrival minutes for at least one tray.")
    if list(arrivals) != sorted(arrivals) or min(scan, label) <= 0:
        raise ValueError("Order trays by arrival and use positive durations.")
    scan_free = label_free = 0
    trace = []
    for tray, arrival in enumerate(arrivals, start=1):
        s0 = max(arrival, scan_free)
        s1 = s0 + scan
        l0 = max(s1, label_free)
        l1 = l0 + label
        trace.append((tray, arrival, s0, s1, l0, l1))
        scan_free, label_free = s1, l1
    return trace

trace = simulate([0, 2, 9])
assert trace == [(1, 0, 0, 4, 4, 11),
                 (2, 2, 4, 8, 11, 18),
                 (3, 9, 9, 13, 18, 25)]
for row in trace:
    print(row)

(1, 0, 0, 4, 4, 11)
(2, 2, 4, 8, 11, 18)
(3, 9, 9, 13, 18, 25)


**Discuss** · Could a simulator that forces the scanner to wait for labeling of the preceding tray reproduce all three trace rows?

No. Tray 2 begins scanning at minute 4, while tray 1 is being labeled. A program that serializes the entire two-station workflow would already disagree on the second tray.

## Run one intervention with the other inputs fixed

Ask when the batch would finish if every scan took two minutes. The runner receives the named change, checks whether it is supported and calls the supplied function with the same arrival list. Keep the original and modified traces for comparison. This is a hand-written exercise in evaluating tool use by a language model. The intervention and simulator are supplied so you can inspect the answer without generating or executing new code.

Δ = Fₙ(x; s′, b) − Fₙ(x; s, b)

x is the ordered vector of tray arrivals, n the number of trays, s′ the proposed scan duration, and b the unchanged labeling duration. A negative Δ means earlier completion. For arrivals 0, 2 and 9, changing s from 4 to 2 moves the last finish from 25 to 23.

In [3]:
def simulate(arrivals, scan=4, label=7):
    if not arrivals or any(t < 0 for t in arrivals):
        raise ValueError("Provide nonnegative arrival minutes for at least one tray.")
    if list(arrivals) != sorted(arrivals) or min(scan, label) <= 0:
        raise ValueError("Order trays by arrival and use positive durations.")
    scan_free = label_free = 0
    trace = []
    for tray, arrival in enumerate(arrivals, start=1):
        s0 = max(arrival, scan_free)
        s1 = s0 + scan
        l0 = max(s1, label_free)
        l1 = l0 + label
        trace.append((tray, arrival, s0, s1, l0, l1))
        scan_free, label_free = s1, l1
    return trace

def run_change(arrivals, field, value):
    if field not in {"scan", "label", "delay_last"}:
        return None  # The supplied clauses do not define this change.
    if field == "delay_last":
        if value < 0:
            raise ValueError("Delay must be nonnegative.")
        changed = [*arrivals[:-1], arrivals[-1] + value]
        return simulate(changed)
    settings = {field: value}
    return simulate(arrivals, **settings)

arrivals = [0, 2, 9]
before = simulate(arrivals)
after = run_change(arrivals, "scan", 2)
assert arrivals == [0, 2, 9]
assert [row[-1] for row in after] == [9, 16, 23]
assert run_change(arrivals, "extra_labeler", 1) is None
print("Last finish:", before[-1][-1], "->", after[-1][-1])
print("Change in minutes:", after[-1][-1] - before[-1][-1])

Last finish: 25 -> 23
Change in minutes: -2


**Discuss** · Why does cutting three scan times by two minutes move the final finish by only two minutes?

The faster scanner lets the first tray reach labeling two minutes earlier. In both runs the labeler then works continuously through all three trays. Further scanning savings leave subsequent trays waiting for that station.

## Independent task

For the same arrivals, compare two separate proposals: two-minute scans or three-minute labels. Predict the three label-finish times under each proposal before running the tool. Identify the first event where the traces differ from the historical run, then give a clause that explains the difference.

In [4]:
def simulate(arrivals, scan=4, label=7):
    if not arrivals or any(t < 0 for t in arrivals):
        raise ValueError("Provide nonnegative arrival minutes for at least one tray.")
    if list(arrivals) != sorted(arrivals) or min(scan, label) <= 0:
        raise ValueError("Order trays by arrival and use positive durations.")
    scan_free = label_free = 0
    trace = []
    for tray, arrival in enumerate(arrivals, start=1):
        s0 = max(arrival, scan_free)
        s1 = s0 + scan
        l0 = max(s1, label_free)
        l1 = l0 + label
        trace.append((tray, arrival, s0, s1, l0, l1))
        scan_free, label_free = s1, l1
    return trace

def run_change(arrivals, field, value):
    if field not in {"scan", "label", "delay_last"}:
        return None  # The supplied clauses do not define this change.
    if field == "delay_last":
        if value < 0:
            raise ValueError("Delay must be nonnegative.")
        changed = [*arrivals[:-1], arrivals[-1] + value]
        return simulate(changed)
    settings = {field: value}
    return simulate(arrivals, **settings)

arrivals = [0, 2, 9]
print("Historical trace:", simulate(arrivals))
for field, value in [("scan", 2), ("label", 3)]:
    trace = run_change(arrivals, field, value)
    # Extract all label finishes and compare the first changed event.
    print(field, "first row", trace[0])

Historical trace: [(1, 0, 0, 4, 4, 11), (2, 2, 4, 8, 11, 18), (3, 9, 9, 13, 18, 25)]
scan first row (1, 0, 0, 2, 2, 9)
label first row (1, 0, 0, 4, 4, 7)


## Compare your solution

Column 3, counting from zero, is the scan end. Column 5 is the label end. Compare the first tray under each change before examining waiting times on later trays.

In [5]:
def simulate(arrivals, scan=4, label=7):
    if not arrivals or any(t < 0 for t in arrivals):
        raise ValueError("Provide nonnegative arrival minutes for at least one tray.")
    if list(arrivals) != sorted(arrivals) or min(scan, label) <= 0:
        raise ValueError("Order trays by arrival and use positive durations.")
    scan_free = label_free = 0
    trace = []
    for tray, arrival in enumerate(arrivals, start=1):
        s0 = max(arrival, scan_free)
        s1 = s0 + scan
        l0 = max(s1, label_free)
        l1 = l0 + label
        trace.append((tray, arrival, s0, s1, l0, l1))
        scan_free, label_free = s1, l1
    return trace

def run_change(arrivals, field, value):
    if field not in {"scan", "label", "delay_last"}:
        return None  # The supplied clauses do not define this change.
    if field == "delay_last":
        if value < 0:
            raise ValueError("Delay must be nonnegative.")
        changed = [*arrivals[:-1], arrivals[-1] + value]
        return simulate(changed)
    settings = {field: value}
    return simulate(arrivals, **settings)

arrivals = [0, 2, 9]
old = simulate(arrivals)
names = ["tray", "arrival", "scan start", "scan end", "label start", "label end"]
for field, value, expected in [("scan", 2, [9, 16, 23]),
                               ("label", 3, [7, 11, 16])]:
    trace = run_change(arrivals, field, value)
    assert [row[-1] for row in trace] == expected
    first_change = next((new[0], names[j], a, b)
                        for prior, new in zip(old, trace)
                        for j, (a, b) in enumerate(zip(prior, new)) if a != b)
    print(field, first_change, expected, "delta", expected[-1] - old[-1][-1])
assert simulate(arrivals) == old

scan (1, 'scan end', 4, 2) [9, 16, 23] delta -2
label (1, 'label end', 11, 7) [7, 11, 16] delta -9


The scan proposal first changes tray 1’s scan end from 4 to 2 under the modified P2 duration. The label proposal first changes its label end from 11 to 7 under the modified P3 duration. Their separate batch-level changes are −2 and −9 minutes. These are results for the fictional process under its stated assumptions.

## Mentor review

Bring one source-to-rule table, the historical trace, and both intervention traces. Ask the mentor to approve the runner’s supported changes for a comparison or name the specific clause-to-code mapping that needs revision.

- Every change names its clause, parameter, unit and untouched inputs.
- A hand calculation checks at least one waiting event at each station.
- The runner reports an unsupported change explicitly and the conclusion is scoped to this process.
- What source passage licenses simultaneous work at the two stations?
- What additional rule would the tool need before it could answer a question about a second labeler?

Freeze the museum packet, rule mapping and tested runner. Carry the ordered arrival list, named intervention and per-tray trace into Module 04, where retrieval and execution will face the same new batches.